In [0]:
%pip install geopandas
%pip install pyrosm

  Obtaining dependency information for geopandas from https://files.pythonhosted.org/packages/3c/78/6a04792ace63a93e162f1305392d500ae8ddcb620e7eb88a22fd622b35bb/geopandas-1.1.3-py3-none-any.whl.metadata
  Obtaining dependency information for numpy>=1.24 from https://files.pythonhosted.org/packages/cf/c5/9fcb7e0e69cef59cf10c746b84f7d58b08bc66a6b7d459783c5a4f6101a6/numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for pyogrio>=0.7.2 from https://files.pythonhosted.org/packages/89/a4/0aef5837b4e11840f501e48e01c31242838476c4f4aff9c05e228a083982/pyogrio-0.12.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for pandas>=2.0.0 from https://files.pythonhosted.org/packages/20/17/ec40d981705654853726e7ac9aea9ddbb4a5d9cf54d8472222f4f3de06c2/pandas-3.0.2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/79.5 kB ? eta -:--:--
     ━

In [0]:
# Azure Storage 연결을 위한 SAS 토큰 설정
STORAGE_ACCOUNT = ""
CONTAINER = ""
SAS_TOKEN = ""


# Spark 설정에 SAS 토큰 등록
spark.conf.set(f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net", SAS_TOKEN)

print("스토리지 연결 준비 완료")

스토리지 연결 준비 완료


In [0]:
# Databricks 경로
MOUNT_POINT = f"/mnt/{CONTAINER}"
# Azure Blob 스토리지 경로
SOURCE = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"
# SAS토큰으로 인증
CONF_KEY = f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net"

# 2. 이미 마운트되어 있는지 확인 후, 안 되어있으면 마운트 실행
if not any(mount.mountPoint == MOUNT_POINT for mount in dbutils.fs.mounts()):
    dbutils.fs.mount(
        source = SOURCE,
        mount_point = MOUNT_POINT,
        extra_configs = {CONF_KEY: SAS_TOKEN}
    )
    print(f"마운트 성공! 이제 /dbfs{MOUNT_POINT} 경로로 접근 가능합니다.")
else:
    print(f"이미 마운트되어 있습니다: {MOUNT_POINT}")

이미 마운트되어 있습니다: /mnt/raw


In [0]:
"""
강남/강동/송파 권역 OSM 공원 데이터 정제 및 도로 네트워크 추출 파이프라인
(최초 생성 및 적재 버전)
"""
import re
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import box
from pyrosm import OSM

# ==========================================
# 0. 초기 설정
# ==========================================
SE_SEOUL_BBOX = {
    "min_lon": 127.01, "min_lat": 37.42,
    "max_lon": 127.19, "max_lat": 37.58,
}

bbox_poly = box(
    SE_SEOUL_BBOX["min_lon"], SE_SEOUL_BBOX["min_lat"],
    SE_SEOUL_BBOX["max_lon"], SE_SEOUL_BBOX["max_lat"],
)

PBF_PATH       = "/dbfs/mnt/raw/osm/south-korea-latest.osm.pbf"
BLOB_RAW_PATH  = "/dbfs/mnt/raw/osm"
CSV_BASE_PATH  = "/dbfs/mnt/raw/parks"


# =======================================================
# 1. PBF → BBox edges(도로) 추출 및 저장, edges_final.geojson 저장
# =======================================================
osm = OSM(PBF_PATH, bounding_box=[
    SE_SEOUL_BBOX["min_lon"], SE_SEOUL_BBOX["min_lat"],
    SE_SEOUL_BBOX["max_lon"], SE_SEOUL_BBOX["max_lat"],
])

# 보행 네트워크 추출 및 노이즈 제거
nodes, edges = osm.get_network(network_type="walking", nodes=True)

EDGE_COLUMNS = ["id", "u", "v", "geometry", "length", "surface", "smoothness", "highway"]
save_cols = [c for c in EDGE_COLUMNS if c in edges.columns]

edges[save_cols].to_file(f"{BLOB_RAW_PATH}/edges_final.geojson", driver="GeoJSON")
print(f"edges_final.geojson 저장 완료 ({len(edges):,}건)")

edges_for_spark = edges[save_cols].copy()

# 1. geometry를 WKT(텍스트)로 변환 (Spark는 Geo형태를 바로 인식 못함)
edges_for_spark["geometry"] = edges_for_spark["geometry"].apply(lambda x: x.wkt if x is not None else None)

# 2. Spark 데이터프레임으로 변환
edges_for_spark = edges_for_spark.fillna(pd.NA)
spark_edges_df = spark.createDataFrame(edges_for_spark)

# 3. Delta 테이블로 저장 (이걸 해야 spark.table로 불러올 수 있음)
spark_edges_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.osm_edges")

print("✅ 이제 spark.table('bronze.osm_edges')로 불러올 수 있습니다!")

# ==========================================
# 2. PBF → BBox leisure(공원 등) 데이터 직접 추출 (최초 적재)
# ==========================================
leisures = osm.get_pois(custom_filter={"leisure": True})

if leisures is None or leisures.empty:
    print("해당 구역에 leisure 데이터가 없습니다. 빈 GeoDataFrame으로 진행합니다.")
    leisures = gpd.GeoDataFrame(columns=['name', 'leisure', 'geometry'], crs="EPSG:4326")
else:
    # 필요한 컬럼만 선택
    keep_cols = [c for c in ["name", "leisure", "geometry"] if c in leisures.columns]
    leisures = leisures[keep_cols].copy()

print(f"PBF에서 추출된 초기 leisure 레코드: {len(leisures):,}건")


# ==========================================
# 3. 추출된 데이터 중 타겟 권역 내 OSM 공원 제거
# ==========================================
# 'park' 태그 데이터만 분리
osm_parks = leisures[leisures["leisure"] == "park"].copy()

# BBox 내부의 부정확한 OSM 공원은 제거 (이후 CSV 데이터로 대체)
is_inside_target = osm_parks.geometry.centroid.within(bbox_poly)
osm_parks_filtered = osm_parks[~is_inside_target].copy()

print(f"타 지역 OSM 공원 유지: {len(osm_parks_filtered):,}건")


# ==========================================
# 4. 구청 공식 공원 CSV 로드 함수
# ==========================================
def load_pet_parks(gu_name: str) -> pd.DataFrame:
    file_path = f"{CSV_BASE_PATH}/{gu_name}.csv"
    try:
        df = pd.read_csv(file_path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding="cp949")

    df.columns = df.columns.str.strip()
    df = df.rename(columns={"공원명": "name", "위도": "lat", "경도": "lon", "반려동물": "pet"})

    return df[["name", "lat", "lon", "pet"]]


# ==========================================
# 5. 커스텀 공원 데이터 통합 및 객체화
# ==========================================
park_df = pd.concat([
    load_pet_parks("gangnam"),
    load_pet_parks("gangdong"),
    load_pet_parks("songpa")
], ignore_index=True)

park_df = park_df.dropna(subset=["lat", "lon"]).copy()
park_gdf = gpd.GeoDataFrame(
    park_df,
    geometry=gpd.points_from_xy(park_df["lon"], park_df["lat"]),
    crs="EPSG:4326"
)
park_gdf["leisure"] = "park"

# ==========================================
# 6. 최종 통합 (OSM 타 지역 공원 + 커스텀 공원)
# ==========================================
park_gdf_cleaned = park_gdf.drop(columns=["lat", "lon"])
combined_park_only = pd.concat([osm_parks_filtered, park_gdf_cleaned], ignore_index=True)

print(f"통합 완료: OSM({len(osm_parks_filtered):,}) + 커스텀({len(park_gdf_cleaned):,})")

# ==========================================
# 7. Spark Delta 테이블 저장
# ==========================================
combined_park_only["geometry"] = combined_park_only["geometry"].apply(
    lambda x: x.wkt if x is not None else None
)
combined_park_only = combined_park_only.fillna(pd.NA)

try:
    spark_final_df = spark.createDataFrame(combined_park_only)
    spark_final_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze.osm_leisure")
    
    print(f"✅ 성공: bronze.leisure_clean 최초 적재 완료 (총 {len(combined_park_only):,}건)")
except Exception as e:
    print(f"❌ 저장 실패: {e}")

edges_final.geojson 저장 완료 (224,690건)


/root/.ipykernel/1504/command-6848812055711808-388460430:50: UserWarning: Geometry column does not contain geometry.
  edges_for_spark["geometry"] = edges_for_spark["geometry"].apply(lambda x: x.wkt if x is not None else None)


✅ 이제 spark.table('bronze.osm_edges')로 불러올 수 있습니다!


/local_disk0/.ephemeral_nfs/envs/pythonEnv-671768b8-1fc0-4b05-87cc-b2df3b0ae798/lib/python3.11/site-packages/pyrosm/pois.py:38: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  gdf = prepare_geodataframe(


PBF에서 추출된 초기 leisure 레코드: 3,291건


/root/.ipykernel/1504/command-6848812055711808-388460430:87: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  is_inside_target = osm_parks.geometry.centroid.within(bbox_poly)


타 지역 OSM 공원 유지: 0건
통합 완료: OSM(0) + 커스텀(360)


/root/.ipykernel/1504/command-6848812055711808-388460430:137: UserWarning: Geometry column does not contain geometry.
  combined_park_only["geometry"] = combined_park_only["geometry"].apply(


✅ 성공: bronze.leisure_clean 최초 적재 완료 (총 360건)


In [0]:
from pyspark.sql.functions import col, count, when, isnan, lit, round

# 1. 브론즈 테이블 로드
bronze_df = spark.table("bronze.osm_edges")
total_rows = bronze_df.count()

print(f"총 데이터 개수: {total_rows} 행")

# 2. 컬럼별 결측치 계산 (Spark Native 연산)
null_analysis = bronze_df.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in bronze_df.columns
])

# 3. 결과를 보기 좋게 한 줄씩 출력 (Unpivot 방식)
print("\n🔍 컬럼별 결측치 현황 (비율 순)")

# 분석 결과를 수집해서 리스트로 만듭니다.
result_row = null_analysis.first()
status_data = []

for c in bronze_df.columns:
    null_count = result_row[c]
    print(c, null_count)


총 데이터 개수: 224690 행

🔍 컬럼별 결측치 현황 (비율 순)
id 0
u 0
v 0
geometry 0
length 0
surface 190923
smoothness 223323
highway 0


In [0]:
# 좌표계(5186)에서 4326으로 바꿔서 저장한 코드
import geopandas as gpd
import pandas as pd

DBFS_RAW_DIR = "/dbfs/mnt/raw"  

def save_shp_to_bronze_with_dbfs(sub_path, table_name):
    full_path = f"{DBFS_RAW_DIR}/{sub_path}"
    print(f"데이터 불러오는 중: {full_path}")
    
    gdf = gpd.read_file(full_path)
    print(f"[{table_name}] 원본 CRS: {gdf.crs}")

    if gdf.crs is None:
        print(f"[{table_name}] ⚠️ CRS 없음 → EPSG:5186 강제 설정")
        gdf = gdf.set_crs("EPSG:5186")

    pdf = pd.DataFrame(gdf)
    if 'geometry' in pdf.columns:
        gdf_4326 = gdf.to_crs("EPSG:4326")
        pdf['geometry'] = gdf_4326['geometry'].apply(lambda x: x.buffer(0).wkt if x else None)
        pdf['crs_info'] = "EPSG:4326"

    spark_df = spark.createDataFrame(pdf.where(pd.notnull(pdf), None))
    
    full_table_name = f"bronze.{table_name}"
    spark_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("mergeSchema", "true") \
        .saveAsTable(full_table_name)
    print(f"[{table_name}] ✅ 델타 테이블 저장 완료!\n")

# 데이터 일괄 실행
save_shp_to_bronze_with_dbfs("sig/sig.shp", "sig")
save_shp_to_bronze_with_dbfs("vworld/slope/gangnam_songpa_gangdong_soilslope.shp", "vworld_slope")
save_shp_to_bronze_with_dbfs("vworld/soil_type/gangnam_songpa_gangdong_soil_type.shp", "vworld_soil_type")
save_shp_to_bronze_with_dbfs("vworld/gravel/gangnam_songpa_gangdong_gravel.shp", "vworld_gravel")
save_shp_to_bronze_with_dbfs("vworld/soil_depth/gangnam_songpa_gangdong_soil_depth.shp", "vworld_soil_depth")
save_shp_to_bronze_with_dbfs("vworld/drainage/gangnam_songpa_gangdong_drainage_class.shp", "vworld_drainage")

데이터 불러오는 중: /dbfs/mnt/raw/sig/sig.shp
[sig] 원본 CRS: None
[sig] ⚠️ CRS 없음 → EPSG:5186 강제 설정
[sig] ✅ 델타 테이블 저장 완료!

데이터 불러오는 중: /dbfs/mnt/raw/vworld/slope/gangnam_songpa_gangdong_soilslope.shp
[vworld_slope] 원본 CRS: EPSG:5181
[vworld_slope] ✅ 델타 테이블 저장 완료!

데이터 불러오는 중: /dbfs/mnt/raw/vworld/soil_type/gangnam_songpa_gangdong_soil_type.shp
[vworld_soil_type] 원본 CRS: EPSG:5181
[vworld_soil_type] ✅ 델타 테이블 저장 완료!

데이터 불러오는 중: /dbfs/mnt/raw/vworld/gravel/gangnam_songpa_gangdong_gravel.shp
[vworld_gravel] 원본 CRS: EPSG:5181
[vworld_gravel] ✅ 델타 테이블 저장 완료!

데이터 불러오는 중: /dbfs/mnt/raw/vworld/soil_depth/gangnam_songpa_gangdong_soil_depth.shp
[vworld_soil_depth] 원본 CRS: EPSG:5181
[vworld_soil_depth] ✅ 델타 테이블 저장 완료!

데이터 불러오는 중: /dbfs/mnt/raw/vworld/drainage/gangnam_songpa_gangdong_drainage_class.shp
[vworld_drainage] 원본 CRS: EPSG:5181
[vworld_drainage] ✅ 델타 테이블 저장 완료!



In [0]:
from pyspark.sql.functions import when, col

# 1. 수정된 데이터프레임 만들기
df_fixed = spark.table("bronze.vworld_soil_depth") \
    .withColumn("VLDSOILDEP", when(col("VLDSOILDEP") == "±âÅ¸", "기타").otherwise(col("VLDSOILDEP")))

# 2. 저장 실행
df_fixed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.vworld_soil_depth")

print("데이터가 성공적으로 '기타'로 수정되어 저장되었습니다")

데이터가 성공적으로 '기타'로 수정되어 저장되었습니다
